# 🌾 HUẤN LUYỆN YOLO SEGMENTATION — ROBOFLOW DATASET MỚI

Notebook này kế thừa cấu hình train từ `TRAIN_YOLO_SEGMENT.ipynb` và tải dataset **full-grains-segmentaion / version 1** trực tiếp từ Roboflow.

Trước khi chạy, mở **Colab → Secrets** (biểu tượng chìa khóa), tạo secret `ROBOFLOW_API_KEY`, dán API key Roboflow và bật quyền truy cập cho notebook. API key không được lưu trong notebook hoặc Git.

Luồng thực hiện: mount Drive → cài thư viện → tải dataset → kiểm tra `data.yaml` → train → dự đoán thử bằng ảnh validation/test → sao lưu kết quả vào Drive.

In [ ]:
# Bước 1: Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Bước 2: Cài đặt thư viện
%pip install -q ultralytics matplotlib pyyaml roboflow

In [ ]:
# Bước 3: Import
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from google.colab import userdata
from roboflow import Roboflow
from ultralytics import YOLO

In [ ]:
# Bước 4: Tải dataset Roboflow mới
ROBOFLOW_WORKSPACE = "minh-tr-tklp0"
ROBOFLOW_PROJECT = "full-grains-segmentaion"
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = "yolov8"

try:
    roboflow_api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Không đọc được Colab Secret ROBOFLOW_API_KEY. "
        "Hãy tạo secret và bật quyền truy cập cho notebook."
    ) from exc

if not roboflow_api_key:
    raise RuntimeError("Colab Secret ROBOFLOW_API_KEY đang trống.")

os.chdir("/content")
rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT)

DATASET_DIR = Path(dataset.location).resolve()
DATA_YAML = DATASET_DIR / "data.yaml"
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Không tìm thấy data.yaml tại {DATA_YAML}")

print(f"✅ Dataset: {DATASET_DIR}")
print(f"✅ Cấu hình: {DATA_YAML}")

In [ ]:
# Bước 5: Cấu hình train và kiểm tra dataset
BASE_PATH = Path("/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES")
MODEL_NAME = "yolov8s-seg.pt"
EPOCHS = 100
BATCH_SIZE = 8
IMG_SIZE = 640
DEVICE = 0

MODELS_DIR = BASE_PATH / "MODELS"
RUN_ROOT = Path("/content/rice_seed_segmentation")
DATASET_TAG = f"{ROBOFLOW_PROJECT}-v{ROBOFLOW_VERSION}"
TRAIN_EXP_NAME = f"{DATASET_TAG}_{Path(MODEL_NAME).stem}_trained"
LOCAL_RESULT_DIR = RUN_ROOT / TRAIN_EXP_NAME
DRIVE_SAVE_DIR = BASE_PATH / "RESULTS" / TRAIN_EXP_NAME

with DATA_YAML.open(encoding="utf-8") as stream:
    data_config = yaml.safe_load(stream)

required_splits = ["train", "val"]
missing_splits = [key for key in required_splits if not data_config.get(key)]
if missing_splits:
    raise ValueError(f"data.yaml thiếu split bắt buộc: {missing_splits}")

print(f"Tên thí nghiệm : {TRAIN_EXP_NAME}")
print(f"Model khởi tạo : {MODEL_NAME}")
print(f"Epoch / batch  : {EPOCHS} / {BATCH_SIZE}")
print(f"Classes        : {data_config.get('names')}")
print(f"Lưu Drive      : {DRIVE_SAVE_DIR}")

In [ ]:
# Bước 6: Huấn luyện YOLO Segmentation
drive_model_path = MODELS_DIR / MODEL_NAME
model = YOLO(str(drive_model_path) if drive_model_path.exists() else MODEL_NAME)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    project=str(RUN_ROOT),
    name=TRAIN_EXP_NAME,
    exist_ok=True,
    patience=30,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=10,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.5,
    close_mosaic=10,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cache="ram",
    amp=True,
    deterministic=True,
    seed=0,
)

BEST_WEIGHT_LOCAL = LOCAL_RESULT_DIR / "weights" / "best.pt"
if not BEST_WEIGHT_LOCAL.is_file():
    raise FileNotFoundError(f"Train kết thúc nhưng không tìm thấy {BEST_WEIGHT_LOCAL}")
print(f"✅ Best weight: {BEST_WEIGHT_LOCAL}")

In [ ]:
# Bước 7: Dự đoán thử trên một ảnh validation/test của chính dataset
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
candidate_dirs = [DATASET_DIR / "valid" / "images", DATASET_DIR / "test" / "images"]
test_image = next(
    (path for folder in candidate_dirs if folder.is_dir()
     for path in sorted(folder.iterdir()) if path.suffix.lower() in image_extensions),
    None,
)

if test_image is None:
    print("⚠️ Dataset không có ảnh valid/test để dự đoán nhanh.")
else:
    trained_model = YOLO(str(BEST_WEIGHT_LOCAL))
    prediction = trained_model.predict(
        source=str(test_image),
        conf=0.4,
        save=True,
        project=str(RUN_ROOT),
        name=f"{TRAIN_EXP_NAME}_prediction",
        exist_ok=True,
    )
    print(f"✅ Đã dự đoán thử: {test_image}")

In [ ]:
# Bước 8: Sao lưu kết quả train vào Google Drive
if not LOCAL_RESULT_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục kết quả {LOCAL_RESULT_DIR}")

DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RESULT_DIR, DRIVE_SAVE_DIR, dirs_exist_ok=True)
print("=" * 60)
print("🎉 Đã sao lưu kết quả train vào Google Drive")
print(f"📂 {DRIVE_SAVE_DIR}")
print(f"⭐ {DRIVE_SAVE_DIR / 'weights' / 'best.pt'}")
print("=" * 60)

In [ ]:
# Bước 9: Hiển thị biểu đồ kết quả và confusion matrix
plot_files = [
    (DRIVE_SAVE_DIR / "results.png", "Loss & mAP qua các epoch"),
    (DRIVE_SAVE_DIR / "confusion_matrix_normalized.png", "Confusion matrix chuẩn hóa"),
]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for axis, (image_path, title) in zip(axes, plot_files):
    if image_path.is_file():
        axis.imshow(plt.imread(image_path))
        axis.set_title(title, fontsize=12, fontweight="bold")
    else:
        axis.text(0.5, 0.5, f"Không tìm thấy\n{image_path.name}", ha="center", va="center")
    axis.axis("off")

plt.tight_layout()
plt.show()